# Notebook 2: Mathematical Specification and Scalar Forward Propagation
## Neural Network from Scratch: Binary MNIST Classification

## 1. Notebook Overview
In Notebook 1, we prepared the MNIST dataset for binary classification by retaining only images representing the digits (0) and (1). We also reshaped and normalized the images so that they could eventually be passed into a neural network.


Before training a neural network, however, we need to understand exactly how a prediction is computed.


This notebook focuses exclusively on **forward propagation**, which is the process of moving information from the input layer through the hidden layer and finally to the output layer.


We will intentionally begin with scalar calculations and explicit Python loops rather than immediately using NumPy matrix multiplication. Although loops are computationally inefficient, they make every multiplication and addition visible.


The notebook follows this sequence:
1. Define a one-hidden-layer neural-network architecture.
2. specify the dimensions of every input, weight, bias, and activation;
3. derive the forward-propagation equations;
4. implement one neuron using scalar operations;
5. extend the implementation to an entire hidden layer using loops;
6. manually verify the loop implementation on a small synthetic example;
7. confirm the loop implementation using vectorized NumPy calculations;
8. apply the scalar implementation to one MNIST image.
   
At this stage, the network will use randomly initialized parameters. **Therefore, it is not expected to classify MNIST images accurately yet.**


The goal is to understand the computation that produces a prediction.

## 2. Import Required Libraries
We will use:
- NumPy for numerical operations and parameter storage;
- Matplotlib for displaying an MNIST image.

Although NumPy supports efficient matrix multiplication, the primary implementation in this notebook will use explicit loops.

In [1]:
import numpy as np 
import matplotlib.pyplot as plt

## 3. Set a Random Seed
### Explanation
Neural-network weights are usually initialized randomly.
Without a fixed random seed, the initial weights would change whenever the notebook is rerun. That would cause the network's initial outputs to change as well.

Setting a seed gives us reproducible results:
$$
\text{same random seed}
\Longrightarrow
\text{same initial weights}
\Longrightarrow
\text{same initial predictions}.
$$
Reproducibility will be especially important later when we compare:
- scalar Python loops;
- vectorized NumPy operations;
- a TensorFlow implementation.

# Part I: Mathematical Specification
## 4. Define the Binary Classification Problem
Each original MNIST image contains $(28 \times 28)$ grayscale pixels.
After flattening an image, one observation becomes a vector containing
$
28 \times 28 = 784
$
input values.
We write one input image as
$$
\begin{bmatrix}
x_1 \
\\
x_2 \
\\
\vdots \
\\
x_{784}
\end{bmatrix}
\in \mathbb{R}^{784}.
$$
Each normalized pixel value satisfies
$$
0 \leq x_j \leq 1.
$$
The target variable is
$$
y =
\begin{cases}
0, & \text{if the image represents digit 0},\
\\
1, & \text{if the image represents digit 1}.
\end{cases}
$$
The neural network will produce an output
$$
\hat{y} \in (0,1),
$$
which we interpret as
$$
P(y=1 \mid \mathbf{x}).
$$
For example:
- $\hat{y}=0.03$ means the model assigns a low probability to digit (1);
- $\hat{y}=0.92$ means the model assigns a high probability to digit (1).
  
A decision threshold of (0.5) can later convert this probability into a predicted class:
$$
\begin{cases}
0, & \hat{y}<0.5,\
\\
1, & \hat{y}\geq 0.5.
\end{cases}
$$

## 5. Define a One-Hidden-Layer Architecture
We will use the following neural-network architecture:
$$
784 \longrightarrow H \longrightarrow 1.
$$
The three layers are:
1. an input layer containing 784 pixel values;
2. one hidden layer containing (H) neurons;
3. one output neuron for binary classification.

For the initial MNIST implementation, we will choose
$
H=8.
$
Therefore, the architecture is
$$
784 \longrightarrow 8 \longrightarrow 1.
$$

This is intentionally a small network.
The hidden layer is large enough to demonstrate how multiple neurons work together, while remaining small enough for us to inspect individual calculations.

In [3]:
input_size = 784 
hidden_size = 8 
output_size = 1 
print("Input units:", input_size) 
print("Hidden units:", hidden_size) 
print("Output units:", output_size)

Input units: 784
Hidden units: 8
Output units: 1


## 6. Layer Notation
We use superscripts to identify the neural-network layer.
- Layer (0): input layer
- Layer (1): hidden layer
- Layer (2): output layer

The input vector is also treated as the activation of layer (0):
$$
\mathbf{a}^{(0)}=\mathbf{x}.
$$

For the hidden layer, we first calculate the linear component:

$$
\mathbf{W}^{(1)}\mathbf{x}
+
\mathbf{b}^{(1)}.
$$
We then apply an activation function:

$$
\sigma\left(\mathbf{z}^{(1)}\right).
$$
For the output layer:

$$
\mathbf{W}^{(2)}\mathbf{a}^{(1)}
+
b^{(2)}.
$$
Finally,

$$
\sigma\left(z^{(2)}\right).
$$
Because there is only one output neuron, we identify the final activation with the prediction:
$$
\hat{y}=a^{(2)}.
$$

## 7. Difference Between (z) and (a)
For every neuron, two values are important.

### Pre-activation value

The pre-activation value is the weighted sum before applying the activation function:
$$
\sum_j w_jx_j+b.
$$

### Activation value

The activation value is the output after applying the activation function:
$$
a=\sigma(z).
$$
Therefore, a neuron performs two conceptual steps:
$$
\text{weighted sum}
\longrightarrow
\text{activation function}.
$$
More explicitly,
$$
x_1,\ldots,x_D
\quad\longrightarrow\quad
z
\quad\longrightarrow\quad
a.
$$
The distinction will become important during backpropagation because gradients pass through both the linear operation and the activation function.